In [92]:
from plotly.io import show
from sklearn.model_selection import train_test_split
from skfolio import Population

from skfolio.optimization import (
    EqualWeighted, 
    MaximumDiversification,
    Random
)
from skfolio.preprocessing import prices_to_returns
from openbb import obb

sectors = [
    "XLE", 
    "XLF", 
    "XLU", 
    "XLI", 
    "GDX", 
    "XLK", 
    "XLV", 
    "XLY", 
    "XLP", 
    "XLB", 
    "XOP", 
    "IYR", 
    "XHB", 
    "ITB", 
    "VNQ", 
    "GDXJ", 
    "IYE", 
    "OIH", 
    "XME", 
    "XRT", 
    "SMH", 
    "IBB", 
    "KBE", 
    "KRE", 
    "XTL", 
]

index=["SPY"]

In [93]:
SPX = obb.equity.price.historical(
    index, 
    start_date="2020-01-01", 
    provider="yfinance"
).to_df().dropna()

SPX=SPX['close']
SPX = SPX.to_frame('SPX')

print(SPX)

               SPX
date              
2020-01-02  324.87
2020-01-03  322.41
2020-01-06  323.64
2020-01-07  322.73
2020-01-08  324.45
...            ...
2024-01-29  491.27
2024-01-30  490.89
2024-01-31  482.88
2024-02-01  489.20
2024-02-02  494.35

[1029 rows x 1 columns]


In [94]:
SPX_return = prices_to_returns(SPX)

index_train, index_test = train_test_split(
    SPX_return, 
    test_size=0.33, 
    shuffle=False
)

index = EqualWeighted()
index.fit(index_train)
index_bench_train = index.predict(index_train)

index_bench_test = index.predict(index_test)

Download the historic price data, manipulate the DataFrame to use with skfolio, and split the data into training and testing sets.

In [95]:
df = obb.equity.price.historical(
    sectors, 
    start_date="2020-01-01", 
    provider="yfinance"
).to_df()

pivoted = df.pivot(
    columns="symbol", 
    values="close"
).dropna()

X = prices_to_returns(pivoted)

X_train, X_test = train_test_split(
    X, 
    test_size=0.33, 
    shuffle=False
)

Build the models
The next step is to fit different models to the data. We’ll use skfolio to create three separate portfolios: maximum diversification, equal weighted, and random weighted.

For each of the models, we instantiate the skfolio class using the training data. Then we fit the data and create the predictions. Finally, we display the weighted average of volatility for each asset divided by the portfolio volatility to compare diversification of the portfolios.

In [96]:
model = MaximumDiversification()
model.fit(X_train)
ptf_model_train = model.predict(X_train)

bench = EqualWeighted()
bench.fit(X_train)
ptf_bench_train = bench.predict(X_train)

random = Random()
random.fit(X_train)
ptf_random_train = random.predict(X_train)

print(f"Maximum Diversification: {ptf_model_train.diversification:0.2f}")
print(f"Equal Weighted model: {ptf_bench_train.diversification:0.2f}")
print(f"Random Weighted model: {ptf_random_train.diversification:0.2f}")

Maximum Diversification: 1.48
Equal Weighted model: 1.30
Random Weighted model: 1.24


Predict the portfolio weights

In [97]:
ptf_model_test = model.predict(X_test)
ptf_bench_test = bench.predict(X_test)
ptf_random_test = random.predict(X_test)


population = Population([  #function in skfolio
    ptf_model_test, 
    ptf_bench_test, 
    ptf_random_test,
    #index_bench_test
])

population.plot_composition()

Generate the cumulative returns of each strategy to visualize how they performed over the analysis period.

In [98]:
population_ = Population([  #function in skfolio
    ptf_model_test, 
    ptf_bench_test, 
    ptf_random_test,
    index_bench_test
])

In [99]:
population_.plot_cumulative_returns()

In [100]:
population_.summary()

,MaximumDiversification,EqualWeighted,Random,EqualWeighted
Mean,0.055%,0.078%,0.086%,0.095%
Annualized Mean,14.01%,19.83%,21.85%,24.22%
Variance,0.015%,0.014%,0.017%,0.010%
Annualized Variance,3.78%,3.45%,4.30%,2.64%
Semi-Variance,0.0068%,0.0062%,0.0080%,0.0047%
Annualized Semi-Variance,1.72%,1.57%,2.03%,1.21%
Standard Deviation,1.22%,1.16%,1.30%,1.02%
Annualized Standard Deviation,19.44%,18.58%,20.73%,16.26%
Semi-Deviation,0.82%,0.79%,0.89%,0.69%
Annualized Semi-Deviation,13.12%,12.55%,14.24%,10.99%
